
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Exploring Data Transformation in Databricks

This notebook demonstrates the **Medallion Architecture** for data transformation, using **Materialized Views (MV)** and **Streaming Tables (ST)** in SQL. The pipeline progresses through the **Bronze**, **Silver**, and **Gold** layers, showcasing how to build efficient data pipelines in Databricks.

**Learning Objectives**

By the end of this notebook, you should be able to:
- Understand the **Medallion Architecture** and its role in data pipelines.
- Declare and configure **DLT** pipelines for automated data processing.
- Use **Materialized Views** and **Streaming Tables** for different data transformation workloads.
- Enforce data quality with **constraints** in DLT pipelines.
- Explore and analyze tables generated by a DLT pipeline using SQL.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
    - In the drop-down, select **More**.
    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
    
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

- To run this notebook, you need to use one of the following Databricks runtime(s): `17.3.x-scala2.13`

##Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>


```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run ../Includes/Classroom-Setup-3.2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Loading batch 1 of 31...1 seconds


True

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")

Username:          labuser12730509_1763721946@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12730509_1763721946
Working Directory: /Volumes/dbacademy/ops/labuser12730509_1763721946@vocareum_com


## A. Tables as Query Results

DLT adapts standard SQL queries to combine DDL (data definition language) and DML (data manipulation language) into a unified declarative syntax.

There are two distinct types of persistent tables that can be created with DLT:

* **Materialized View**  
Materialized views are refreshed according to the update schedule of the pipeline in which they’re contained. Materialized views are powerful because they can handle any changes in the input. Each time the pipeline updates, query results are recalculated to reflect changes in upstream datasets that might have occurred because of compliance, corrections, aggregations, or general CDC.

* **Streaming Tables**  
Streaming tables allow you to process a growing dataset, handling each row only once. Because most datasets grow continuously over time, streaming tables are good for most ingestion workloads. Streaming tables are optimal for pipelines that require data freshness and low latency.

Note that both of these objects are persisted as tables stored with the Delta Lake protocol (providing ACID transactions, versioning, and many other benefits). We'll talk more about the differences between materialized views and streaming tables later in the notebook.

For both kinds of tables, DLT takes the approach of a slightly modified CTAS (create table as select) statement. Engineers just need to worry about writing queries to transform their data, and DLT handles the rest.

The basic syntax for a SQL DLT query is:

**`CREATE OR REFRESH [STREAMING] TABLE table_name`**<br/>
**`AS select_statement`**<br/>

## B. Explore Available Raw Files

Complete the following steps to explore the available raw data files that will be used for the DLT pipeline:

1. Navigate to the available catalogs by selecting the catalog icon directly to the left of the notebook (do not select the **Catalog** text in the far left navigation bar).
1. Expand the **dbacademy > {schema_name} > Volumes**. (You can find your schema name in the output of the cell you just executed above.)
1. Expand the **stream-source** directory. Notice that the directory contains two subdirectories: **customers** and **orders**.
1. Expand each subdirectory. Notice that each contains a JSON file (00.json) with raw data. We will create a DLT pipeline that will ingest the files within this volume to create tables and materialized views for our consumers.

## C. DLT Pipeline: Customer Order Pipeline

Check out how to create **Streaming Tables** and **Materialized Views (MV)** for the Medallion Architecture by reviewing the **Customer Order Pipeline** notebook. Follow these steps:

1. Open the [Customer Order Pipeline]($./Pipelines/Customer%20Order%20Pipeline) notebook.
   - Do not attempt to execute the code directly. It is intended to be executed within the context of the DLT pipeline workflow.
   - Examine how **Streaming Tables** and **Materialized Views** are implemented to process data through the **Bronze**, **Silver**, and **Gold** layers.
   - Observe the step-by-step creation and transformation of tables within the Medallion Architecture, including data ingestion, validation, and enrichment techniques.
1. After reviewing the pipeline notebook and understanding its concepts, close the tab and return to this notebook to proceed with generating the DLT pipeline and completing additional tasks.

## D. Generate Pipeline Configuration
DLT pipelines can be written in either SQL or python. In the code cell below, note that we are first going to look at the SQL example. 

We are going to manually configure a pipeline using the DLT UI. Configuring this pipeline will require parameters unique to a given user. Run the cell to print out values you'll use to configure your pipeline in subsequent steps.

In [0]:
%sql
-- Set schema for the session
USE SCHEMA ${DA.schema_name};

In [0]:
pipeline_language = "SQL"

DA.print_pipeline_config(pipeline_language)

labuser12730509_1763721946: Data Transformation Pipeline


Pipeline Name:,
Notebook #1 Path:,
Default Catalog:,
Default Schema:,
Source:,


## Create and Configure a Pipeline

Complete the following to configure the pipeline.

Steps:
1. Open the [Pipelines user interface](/pipelines) (or use the **Jobs & Pipelines** option from the left sidebar and select the Jobs & pipelines tab).
1. Click **Create** in the upper-right corner, and select **ETL pipeline** from the dropdown menu.
1. Click on the **Lakeflow Pipelines Editor** option at the top of the job, and toggle the switch to turn **off** the Lakeflow Pipelines UI.

**📌 Note:** A `Try the new Lakeflow Pipelines Editor` pop-up may appear on your screen. Please select **Maybe Later** to proceed with this course.

3. Configure the pipeline as specified below. You'll need the values provided in the cell output above for this step.

| Setting | Instructions |
|--|--|
| Pipeline name | Enter the **Pipeline Name** provided above |
| Cluster | Click on **Serverless** |
| Pipeline mode | Choose **Triggered** |
| Paths | Use the navigator to select or enter the notebook path provided above |
| Storage options | Check for **Unity Catalog** is Enabled  |
| Default catalog | Choose your **Default Catalog** provided above |
| Default schema | Choose the **Default Schema** provided above |
| Configuration | Click **Add Configuration** and input the **Key** and **Value** in the table below|
| Channel | Choose **Current** |


| Key                 | Value                                      |
| ------------------- | ------------------------------------------ |
| **`source`** | Enter the **source** provided above |

<br>

4. Click the **Create** button.
5. Verify that the pipeline mode is set to **Development**.

## Check Your Pipeline Configuration

1. In the Databricks workspace, open the Pipelines UI.
1. Select your pipeline configuration.
1. Review the pipeline configuration settings to ensure they are correctly configured according to the provided instructions.
1. **Important:** Remove the maintenance cluster if it is currently part of your pipeline configuration. This is required to successfully validate the pipeline configuration. Do this by clicking JSON in the upper-right corner and removing the code related to the maintenance cluster.
1. Once you've confirmed that the pipeline configuration is set up correctly and the maintenance cluster has been removed, proceed to the next steps for validating and running the pipeline.

In [0]:
DA.validate_pipeline_config(pipeline_language)

labuser12730509_1763721946: Data Transformation Pipeline
Pipeline validation complete. No errors found.


## Update the pipeline

Trigger an update of the pipeline you created by clicking the **Start** button back in the Pipeline user interface.

##Querying Tables in the Target Database

As long as a target database is specified during DLT Pipeline configuration, tables should be available to users throughout your Databricks environment. Let's explore them now. 

Run the cell below to see the tables registered to the database used so far. The tables were created in the **dbacademy** catalog, within your unique **schema** name.

In [0]:
%sql
SHOW TABLES

database,tableName,isTemporary
labuser12730509_1763721946,customer_bronze,false
labuser12730509_1763721946,customer_bronze_clean,false
labuser12730509_1763721946,customer_order,false
labuser12730509_1763721946,customer_silver,false
labuser12730509_1763721946,customers_ui,false
labuser12730509_1763721946,customers_ui_bronze,false
labuser12730509_1763721946,customers_ui_gold,false
labuser12730509_1763721946,customers_ui_silver,false
labuser12730509_1763721946,order_bronze,false
labuser12730509_1763721946,order_silver,false


Note that the view we defined in our pipeline is absent from our tables list.

Query results from the **`order_silver`** table.

In [0]:
%sql
SELECT * FROM order_silver LIMIT 5;

order_timestamp,customer_id,notifications,order_id,processing_time,source_file
2021-12-25T00:28:12Z,23094,Y,75123,2025-11-21T12:08:10.348Z,00.json
2021-12-25T00:35:00Z,23457,N,75124,2025-11-21T12:08:10.348Z,00.json
2021-12-25T01:14:22Z,23564,Y,75125,2025-11-21T12:08:10.348Z,00.json
2021-12-25T01:34:27Z,23392,N,75126,2025-11-21T12:08:10.348Z,00.json
2021-12-25T02:24:26Z,23101,Y,75127,2025-11-21T12:08:10.348Z,00.json


Recall that **`orders_bronze`** was defined as a streaming table in DLT, but our results here are static.

Because DLT uses Delta Lake to store all tables, each time a query is executed, we will always return the most recent version of the table. But queries outside of DLT will return snapshot results from DLT tables, regardless of how they were defined.

###Show Lineage for Delta Tables in Unity Catalog

Unity Catalog captures runtime data lineage for all **table-to-table operations** executed on Databricks clusters or SQL endpoints. Lineage works seamlessly across all languages, including **SQL, Python, Scala, and R**. It can be visualized in **Data Explorer** in near real-time and can also be retrieved programmatically using the **REST API**. 

![lineage](../Includes/images/lineage.png)

**Lineage Granularity Levels**

Unity Catalog supports data lineage at two levels:
1. **Table-Level Lineage**:
   - Tracks the flow of data between entire tables.
   - Useful for understanding the broader context of data operations.

2. **Column-Level Lineage**:
   - Tracks data transformations at the column level.
   - Ideal for use cases like **GDPR compliance** and tracking sensitive data dependencies.

**Access Control with Table ACLs**

Lineage respects the **Table ACLs** (Access Control Lists) defined in Unity Catalog:
- If a user does not have access to a table in the lineage graph, its details will be redacted.
- However, users will still see the presence of upstream or downstream dependencies, ensuring visibility into the flow of data while maintaining security.

---

**Benefits of Viewing Lineage**
1. **End-to-End Data Visibility**:
   - Understand how data flows through the pipeline, from source to final output.

2. **Compliance and Governance**:
   - Ensure GDPR compliance by tracking sensitive data dependencies at the column level.

3. **Debugging and Optimization**:
   - Identify bottlenecks and optimize transformations for better performance.

### Steps to View Lineage in Unity Catalog

Follow these steps to view the lineage of Delta Tables in Unity Catalog:

**Step 1: Navigate to the Pipelines**
- Run the following code to generate the pipeline URL.
- Click on the printed **Pipeline URL** to navigate to the pipeline page.

**Step 2: Select the Materialized View**
- On the pipeline page, locate the materialized view of interest (e.g., `customer_order`).
- Click on the materialized view to open its **Details** tab. 

**Step 3: Open the Table in Catalog Explorer**
- Under the **Details** tab of the materialized view, locate the table name (e.g., `dbacademy.schema_name.customer_order`).
- Click on the table name to navigate to the **Catalog Explorer**. 

**Step 4: View Lineage Tab**
- In the Catalog Explorer, select the **Lineage** tab from the menu at the top. 
- This will display a summary of the table's upstream and downstream dependencies.

**Step 5: Open the Lineage Graph**
- In the **Lineage** tab, locate the **"See Lineage Graph"** button in the top-right corner of the page.
- Click on the button to open the expanded lineage graph. 

**Step 6: Explore the Lineage Graph**
- The **Lineage Graph** will display the data flow for the table:
   - **Upstream Tables**: Represent the data sources feeding into the pipeline.
   - **Downstream Tables**: Represent the outputs or dependencies created from the table.
- Click on the **`+` icons** to expand the graph and reveal additional details about each connection.

In [0]:
try:
    # Retrieve workspace URL dynamically from Databricks configurations
    workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
    pipeline_name = f"{DA.schema_name}: Data Transformation Pipeline"
    
    # Retrieve and print the clickable pipeline URL
    pipeline_url = get_pipeline_url(workspace_url, pipeline_name)
    print(f"DLT Pipeline URL: {pipeline_url}")
except ValueError as e:
    print(e)

DLT Pipeline URL: https://dbc-72526557-65e2.cloud.databricks.com/pipelines/7bbadfed-be7c-4b9d-91dd-2429aad514d9


## Conclusion

In this notebook, we explored data transformation in Databricks using the Medallion Architecture, highlighting the capabilities of DLT to build robust and efficient pipelines. We demonstrated the creation and configuration of pipelines that use Materialized Views (MV) and Streaming Tables (ST) to process and transform data across Bronze, Silver, and Gold layers.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>